## setup
- Install google genai
- All imports
- Authenticate GCS w/ service account
- Login to wandb

In [ ]:
!pip install -U -q "google-genai"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.8/728.8 kB 14.5 MB/s eta 0:00:00


In [ ]:
from google.cloud import storage
from typing import List, Dict, Tuple
from tqdm import tqdm
import time
import datetime
import json
import os

# basic stuff
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, recall_score, confusion_matrix

# new gemini genai SDK
from google import genai
from google.genai import types
from google.colab import userdata

In [ ]:
# GCS authentication w/ service account
service_account_info_str = userdata.get('GCS_SERVICE_ACCOUNT_KEY')
temp_key_file_path = "/tmp/gcs_service_account_key.json"
with open(temp_key_file_path, "w") as f:
    f.write(service_account_info_str)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = temp_key_file_path

# wandb login
from google.colab import userdata
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_TOKEN')
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: christopher-w-song (christopher-w-song-dougherty-valley-high-school) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## functions

In [ ]:
def classify_with_exemplars(client, model_name: str, classification_prompt: str, classification_type: str, collect_reasoning: bool, temperature: int,
                            tkdname: str, transcription: str = None, exemplars: list = None) -> dict:
    """
    Classify a single sample (text or audio) using Gemini with in-context learning exemplars

    Args:
        client: Google Gen AI client
        model_name: Name of Gemini model to use (e.g., 'gemini-2.5-flash-latest')
        classification_prompt: The prompt to guide the model's classification
        tkdname: The tkdname identifier
        classification_type: "text", "audio", or "multimodal"
        collect_reasoning: Whether to collect reasoning from the model
        transcription: Text transcription (required for text and multimodal classification)
        exemplars: List of exemplar dicts with keys ['tkdname', 'label', 'transcription', 'language']

    Returns:
        Dictionary with classification results
    """

    contents = []

    contents.append(classification_prompt)

    # EXEMPLARS
    contents.append(f"Here are {len(exemplars)} exemplars:")

    for exemplar in exemplars:
        contents.append(f"Label: {exemplar['label']}")
        if classification_type in ["audio", "multimodal"]:
            audio_blob_name = f"audio-data/{exemplar['tkdname']}"
            gcs_uri = f"gs://{bucket_name}/{audio_blob_name}"

            audio_part = types.Part(
                file_data=types.FileData(
                    file_uri=gcs_uri,
                    mime_type="audio/wav"
                )
            )
            contents.append(audio_part)
        if classification_type in ["text", "multimodal"]:
            contents.append(exemplar['transcription'])

    contents.append(f"Here is the sample to be classified:")

    # TARGET SAMPLE
    if classification_type in ["audio", "multimodal"]:
        audio_blob_name = f"flattened-audio-data/{tkdname}"
        gcs_uri = f"gs://{bucket_name}/{audio_blob_name}"

        audio_part = types.Part(
            file_data=types.FileData(
                file_uri=gcs_uri,
                mime_type="audio/wav"
            )
        )
        contents.append(audio_part)
    if classification_type in ["text", "multimodal"]:
        contents.append(transcription)


    start_time = time.time()

    config_dict = {
        "temperature": temperature,
        "response_mime_type": "text/plain"
    }
    if collect_reasoning:
        config_dict["thinking_config"] = types.ThinkingConfig(include_thoughts=True)

    # Generate response with selected contents
    response = client.models.generate_content(
        model=model_name,
        contents=contents,
        config=types.GenerateContentConfig(**config_dict)
    )

    end_time = time.time()
    response_time = round((end_time - start_time), 2)

    # Extract response, reasoning, & finish reason
    reasoning_text = None
    response_text = None
    if response.candidates and response.candidates[0].content.parts:
        for part in response.candidates[0].content.parts:
            if not part.text:
                continue

            if collect_reasoning and hasattr(part, 'thought') and part.thought:
                reasoning_text = part.text
            else:
                response_text = part.text.strip().upper()  # Ensure consistent format

    def clean_response(text):
        """Extract MCI or NC from model response, handling various formats"""
        if not text:
            return "ERROR"

        text = text.strip().upper()

        # Remove common prefixes
        prefixes = ['LABEL:', 'LABEL :', 'DIAGNOSIS:', 'DIAGNOSIS :',
                  'OUTPUT:', 'ANSWER:']
        for prefix in prefixes:
            if prefix in text:
                idx = text.find(prefix)
                text = text[idx + len(prefix):].strip()

        # Extract just the first word if it's a long response
        first_word = text.split()[0] if text.split() else text

        # Only return valid labels
        if 'MCI' in first_word:
            return 'MCI'
        elif 'NC' in first_word:
            return 'NC'
        else:
            return "ERROR"

    response_text = clean_response(response_text)

    finish_reason = response.candidates[0].finish_reason

    # Extract metadata
    usage_metadata = response.usage_metadata
    input_tokens = usage_metadata.prompt_token_count
    output_tokens = usage_metadata.candidates_token_count
    thoughts_tokens = usage_metadata.thoughts_token_count
    total_tokens = usage_metadata.total_token_count

    result = {
        'prediction': response_text,
        'response_time': response_time,
        'finish_reason': finish_reason,
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'thoughts_tokens': thoughts_tokens,
        'total_tokens': total_tokens,
    }

    if collect_reasoning:
        result['reasoning_text'] = reasoning_text

    return result

# helper functions

def save_results(results_df: pd.DataFrame, run_name: str, reasoning_dict: dict = None) -> str:
    """
    Save results to wandb artifacts

    Args:
        results_df: DataFrame with classification results.
        run_name: Base name for the run.
        reasoning_dict: Dictionary with reasoning text.

    Returns:
        String with the results CSV filename that was used.
    """

    # Log results as W&B artifact
    csv_filename = f"results_{run_name}.csv"
    csv_path = f"/content/{csv_filename}"
    results_df.to_csv(csv_path, index=False)

    #csv_path = os.path.join(os.getcwd(), csv_filename)
    #json_path = os.path.join(os.getcwd(), f"reasoning_{run_name}.json")

    results_artifact = wandb.Artifact(f"results_{run_name}", type="results")
    results_artifact.add_file(csv_path)
    wandb.log_artifact(results_artifact)
    print(f"✓ Results logged to W&B")

    # Log reasoning as W&B artifact
    if reasoning_dict is not None:
        json_path = f"/content/reasoning_{run_name}.json"
        with open(json_path, 'w') as f:
            json.dump(reasoning_dict, f, indent=2)
        reasoning_artifact = wandb.Artifact(f"reasoning_{run_name}", type="reasoning")
        reasoning_artifact.add_file(json_path)
        wandb.log_artifact(reasoning_artifact)
        print(f"✓ Reasoning logged to W&B")

    return csv_filename

def evaluate_results(results_df: pd.DataFrame, run_name: str) -> dict:
    """
    Evaluate classification results from a DataFrame

    Args:
        results_df: DataFrame containing the classification results.
        run_name: Base name for the run.

    Returns:
        Dictionary with evaluation metrics.
    """

    from sklearn.metrics import accuracy_score, f1_score, recall_score

    def clean_prediction(pred):
        """Clean prediction to remove prefixes and standardize format"""
        if pd.isna(pred):
            return "ERROR"

        pred = str(pred).strip().upper()

        # Remove common prefixes that the model might add
        prefixes_to_remove = ['LABEL:', 'LABEL :', 'DIAGNOSIS:', 'DIAGNOSIS :']
        for prefix in prefixes_to_remove:
            if pred.startswith(prefix):
                pred = pred[len(prefix):].strip()

        # Only keep valid predictions
        if pred in ['MCI', 'NC']:
            return pred
        else:
            return "ERROR"

    def calculate_subset_metrics(subset_df, subset_name):
        # calculate acc, UAR, micro f1, macro F1 for one subset

        subset_df = subset_df.copy()  # Avoid modifying original
        subset_df['prediction'] = subset_df['prediction'].apply(clean_prediction)

        # filter out errors
        valid_df = subset_df[subset_df['prediction'] != "ERROR"]

        y_true = valid_df['dx'].values
        y_pred = valid_df['prediction'].values

        return {
            "subset": subset_name,
            "samples": len(subset_df),
            "valid_samples": len(valid_df),
            "accuracy": round(accuracy_score(y_true,y_pred), 4),
            "uar": round(recall_score(y_true, y_pred, average='macro', zero_division=0), 4),
            "micro_f1": round(f1_score(y_true, y_pred, average='micro', zero_division=0), 4),
            "macro_f1": round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4)
        }

    all_data = calculate_subset_metrics(results_df, "all_data")
    english_data = results_df[results_df['language'].astype(str).str.lower().isin(['en', 'english'])]
    english_metrics = calculate_subset_metrics(english_data, "english_only")
    chinese_data = results_df[results_df['language'].astype(str).str.lower().isin(['zh', 'chinese', 'cn'])]
    chinese_metrics = calculate_subset_metrics(chinese_data, "chinese_only")

    print(f"\n📊 EVALUATION RESULTS:")
    print(f"{'Subset':<15} {'Valid/Total':<12} {'Accuracy':<10} {'UAR':<10} {'Micro F1':<10} {'Macro F1':<10}")
    print("-" * 75)
    for metrics in [all_data, english_metrics, chinese_metrics]:
        sample_str = f"{metrics['valid_samples']}/{metrics['samples']}"
        print(f"{metrics['subset']:<15} {sample_str:<12} {metrics['accuracy']:<10} {metrics['uar']:<10} {metrics['micro_f1']:<10} {metrics['macro_f1']:<10}")

    # Log to W&B
    wandb.run.summary.update({
        # All data
        "accuracy_all": all_data["accuracy"],
        "uar_all": all_data["uar"],
        "micro_f1_all": all_data["micro_f1"],
        "macro_f1_all": all_data["macro_f1"],
        "valid_samples_all": all_data["valid_samples"],

        # English only
        "accuracy_english": english_metrics["accuracy"],
        "uar_english": english_metrics["uar"],
        "micro_f1_english": english_metrics["micro_f1"],
        "macro_f1_english": english_metrics["macro_f1"],
        "valid_samples_english": english_metrics["valid_samples"],

        # Chinese only
        "accuracy_chinese": chinese_metrics["accuracy"],
        "uar_chinese": chinese_metrics["uar"],
        "micro_f1_chinese": chinese_metrics["micro_f1"],
        "macro_f1_chinese": chinese_metrics["macro_f1"],
        "valid_samples_chinese": chinese_metrics["valid_samples"]
    })

    print("✓ Metrics logged to W&B")

    return {
        "all_data": all_data,
        "english_only": english_metrics,
        "chinese_only": chinese_metrics
    }

## execution code

In [ ]:
# args
bucket_name = "taukadial-25"
groundtruth_path = "groundtruth/groundtruth_combined_lang2.csv"
transcription_path = "text-data/transcription_data_modified.csv"
model_name = "gemini-2.5-pro"
collect_reasoning = False
temperature = 1
classification_prompt = """
Assess the cognitive condition based on the input audio data, where an elderly speaker describes one of three images as part of a clinician-guided task.
Indicate the diagnosis using one of these labels: NC (Normal Cognitive) or MCI (Mild Cognitive Impairment).
Output only "NC" or "MCI" as your response. Do not include any explanation, reasoning, or additional text.
"""
classification_type = "audio" # audio, text, or multimodal, make sure to change the prompt to reflect this
exemplar_iterations = 3
timestamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
run_name = f"incontext_{classification_type}_{model_name}_{timestamp}"

checkpoint_interval = 50
checkpoint_filename = "checkpoint_results.csv"
reasoning_filename = "checkpoint_reasoning.json"

resume_run_id = "ee73t0er" #"j6kvv7o9"

# GCS initialization
storage_client = storage.Client()
gcs_bucket = storage_client.bucket(bucket_name)

# Gemini Client initialization
client = genai.Client(
    #api_key = userdata.get('GOOGLE_API_KEY'),
    vertexai = True,
    project = "grand-store-465802-r4",
    location = "us-west1" #global for 3 pro
)

# wandb initialization
run = wandb.init(
    project = "zero-shot-mci-classification",
    id = resume_run_id,
    resume = "allow",
    name = run_name,
    config = {
        "model_name": model_name,
        #"model_type": "closed_source",
        #"model_family": "gemini",
        "bucket_name": bucket_name,
        "groundtruth_path": groundtruth_path,
        #"samples_processed": 507,  # Update with actual size
        "temperature": temperature,
        "prompt": classification_prompt,
        "exemplars_per_sample": exemplar_iterations
    }
)

# get groundtruth as DataFrame
from io import StringIO
blob = gcs_bucket.blob(groundtruth_path)
content = blob.download_as_text()
results_df = pd.read_csv(StringIO(content))

# load transcription data & create lookup map, tehcnically only needed for text & multimodal
transcription_blob = gcs_bucket.blob(transcription_path)
transcription_content = transcription_blob.download_as_text()
transcription_df = pd.read_csv(StringIO(transcription_content))
transcription_map = transcription_df.set_index('tkdname')['transcription'].to_dict()

# initialize new columns for the results.csv, wide data format
results_df['prediction'] = None
for i in range(exemplar_iterations):
    results_df[f'prediction_{i+1}'] = None
    results_df[f'response_time_{i+1}'] = None
    results_df[f'finish_reason_{i+1}'] = None
    results_df[f'input_tokens_{i+1}'] = None
    results_df[f'output_tokens_{i+1}'] = None
    results_df[f'thoughts_tokens_{i+1}'] = None
    results_df[f'total_tokens_{i+1}'] = None
    results_df[f'nc_exemplar_{i+1}'] = None
    results_df[f'mci_exemplar_{i+1}'] = None

reasoning_dict = {} if collect_reasoning else None

# checkpointing logic, only look for checkpoints if we're resuming a run
start_idx = 0
if run.resumed:
    checkpoint_artifact = run.use_artifact(f"checkpoint:latest") # Use the general 'checkpoint:latest' artifact
    checkpoint_dir = checkpoint_artifact.download()

    # Load CSV checkpoint
    checkpoint_csv_path = os.path.join(checkpoint_dir, checkpoint_filename) # Use the predefined checkpoint_filename
    checkpoint_df = pd.read_csv(checkpoint_csv_path)

    # Load reasoning checkpoint if it exists
    reasoning_json_path = os.path.join(checkpoint_dir, reasoning_filename) # Use the predefined reasoning_filename
    if os.path.exists(reasoning_json_path) and collect_reasoning:
        with open(reasoning_json_path, 'r') as f:
            reasoning_dict = json.load(f)
        print(f"✓ Loaded reasoning for {len(reasoning_dict)} samples")

    # Find where to resume
    incomplete_rows = checkpoint_df[checkpoint_df['prediction'].isna()]
    if len(incomplete_rows) > 0:
        start_idx = incomplete_rows.index[0]
        results_df = checkpoint_df
        print(f"✓ Resuming from sample {start_idx + 1}")
    else:
        print("Checkpoint complete, starting fresh")

for idx, row in tqdm(results_df.iterrows(), total=len(results_df), desc=f"Processing: ", colour='green'):
    if idx < start_idx:
        continue

    tkdname = row['tkdname']
    language = row['language']
    transcription = transcription_map.get(tkdname)

    # filter available pool of exemplars, exemplar != target, and language matches
    available_pool = results_df[
        (results_df['tkdname'] != tkdname) &
        (results_df['language'].astype(str).str.lower() == language.lower())
    ]
    # Separate by class for sampling
    nc_samples = available_pool[available_pool['dx'] == 'NC']
    mci_samples = available_pool[available_pool['dx'] == 'MCI']

    predictions = [] # predictions across 5 exemplars, used for MV

    for i in range(exemplar_iterations):
        # random pick exemplar
        nc_exemplar = nc_samples.sample(n=1).iloc[0]
        mci_exemplar = mci_samples.sample(n=1).iloc[0]

        # track which exemplars were used
        trial_num = i + 1
        results_df.at[idx, f'nc_exemplar_{trial_num}'] = nc_exemplar['tkdname']
        results_df.at[idx, f'mci_exemplar_{trial_num}'] = mci_exemplar['tkdname']

        exemplars = [
            {
                'tkdname': nc_exemplar['tkdname'],
                'label': nc_exemplar['dx'],
                'transcription': transcription_map.get(nc_exemplar['tkdname']),
                'language': nc_exemplar['language']
            },
            {
                'tkdname': mci_exemplar['tkdname'],
                'label': mci_exemplar['dx'],
                'transcription': transcription_map.get(mci_exemplar['tkdname']),
                'language': mci_exemplar['language']
            }
        ]

        # Call classification
        result = classify_with_exemplars(
            client,
            model_name,
            classification_prompt,
            classification_type,
            collect_reasoning,
            temperature,
            tkdname,
            transcription,
            exemplars
        )

        # log numerical results to wandb as graph
        wandb.log({
            "response_time": result['response_time'],
            "input_tokens": result['input_tokens'],
            "output_tokens": result['output_tokens'],
            "thoughts_tokens": result['thoughts_tokens'],
            "total_tokens": result['total_tokens']
        })

        # add predictions to a list for majority vote
        predictions.append(result['prediction'])

        # update results csv with prediction info
        trial_num = i + 1
        results_df.at[idx, f'prediction_{trial_num}'] = result['prediction']
        results_df.at[idx, f'response_time_{trial_num}'] = result['response_time']
        results_df.at[idx, f'finish_reason_{trial_num}'] = result['finish_reason']
        results_df.at[idx, f'input_tokens_{trial_num}'] = result['input_tokens']
        results_df.at[idx, f'output_tokens_{trial_num}'] = result['output_tokens']
        results_df.at[idx, f'thoughts_tokens_{trial_num}'] = result['thoughts_tokens']
        results_df.at[idx, f'total_tokens_{trial_num}'] = result['total_tokens']

        # update reasoning dict with reasoning info
        if collect_reasoning and result.get('reasoning_text'):
          if tkdname not in reasoning_dict:
              reasoning_dict[tkdname] = {}
          reasoning_dict[tkdname][f'iteration_{trial_num}'] = result['reasoning_text']

        time.sleep(1)

    # Majority vote to get final prediction
    from collections import Counter
    counts = Counter(predictions)
    final_prediction = counts.most_common(1)[0][0]
    results_df.at[idx, 'prediction'] = final_prediction

    # set a checkpoint
    # Add this one line to get final checkpoint
    if (idx + 1) % checkpoint_interval == 0 or (idx + 1) == len(results_df):
        # Save CSV checkpoint
        results_df.to_csv(checkpoint_filename, index=False)

        # Save reasoning dict checkpoint
        if reasoning_dict:
            with open(reasoning_filename, 'w') as f:
                json.dump(reasoning_dict, f, indent=2)

        # Create wandb artifact with both files
        checkpoint_artifact = wandb.Artifact("checkpoint", type="checkpoint")
        checkpoint_artifact.add_file(checkpoint_filename, name = "checkpoint_results.csv")
        if reasoning_dict:
            checkpoint_artifact.add_file(reasoning_filename, name = "checkpoint_reasoning.json")
        wandb.log_artifact(checkpoint_artifact)

        print(f"✓ Checkpoint saved at sample {idx + 1}")

    time.sleep(1)

save_results(results_df, run_name, reasoning_dict)
evaluate_results(results_df, run_name)

print(f"\nSummary:")
print(f"  Total samples processed: {len(results_df)}")
print(f"  Valid predictions: {len(results_df[results_df['prediction'] != 'ERROR'])}")
print(f"  Error cases: {len(results_df[results_df['prediction'] == 'ERROR'])}")

# wandb log final summary metrics
total_samples = len(results_df)
total_tokens = sum(results_df[f'total_tokens_{i+1}'].sum() for i in range(exemplar_iterations))
total_runtime = sum(results_df[f'response_time_{i+1}'].sum() for i in range(exemplar_iterations))
wandb.run.summary.update(
    {
        "total_runtime_minutes": total_runtime / 60,
        "total_tokens_used": total_tokens,
        "avg_tokens_per_sample": total_tokens / total_samples,
        "samples_per_minute": total_samples / (total_runtime / 60)
    }
)
wandb.finish()

wandb: Detected [google.genai, mcp] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/
wandb:   1 of 1 files downloaded.  


✓ Resuming from sample 401


Processing:  89%|████████▊ | 449/507 [44:02<51:20, 53.11s/it]

✓ Checkpoint saved at sample 450


Processing:  98%|█████████▊| 499/507 [1:28:22<06:57, 52.18s/it]

...

Processing: 100%|█████████▉| 506/507 [1:34:35<00:53, 53.35s/it]

✓ Checkpoint saved at sample 507


Processing: 100%|██████████| 507/507 [1:35:23<00:00, 11.29s/it]


✓ Results logged to W&B

📊 EVALUATION RESULTS:
Subset          Valid/Total  Accuracy   UAR        Micro F1   Macro F1  
---------------------------------------------------------------------------
all_data        507/507      0.6154     0.6136     0.6154     0.6122    
english_only    246/246      0.6098     0.6441     0.6098     0.6097    
chinese_only    261/261      0.6207     0.6189     0.6207     0.6106    
✓ Metrics logged to W&B

Summary:
  Total samples processed: 507
  Valid predictions: 507
  Error cases: 0


input_tokens,▄▃▃▃▄█▃▃▃▁▂▃▃▃▃▂▅▄▆▇▄▂▂▁▃▆█▅▃▅▂▁▅██▂▄▅▃▅
output_tokens,▁████▁████▁▁█▁█████▁▁▁▁████▁▁▁███▁▁█▁▁██
response_time,▃▁▇▃▄▂▅▅▃█▅▂▃▃▂▂▄▂▄▃▃▃▄▄▃▃▂▂▂▃▂▃▃▆▂▅▂▄▄▃
thoughts_tokens,▄▃▂▆▅▄▃▂▁▂▂▄█▄▃▄▃▃▂▇▄▄▃▃▃▄▄▄▃▃▆▃▂▃▃▄▂▁▂▄
total_tokens,▄▁▄▆▅█▅▇▆▂▆▃▄▄▅▆█▅▄▆▂▅▄▄▂▆▅▄▄▇▄█▄▅▄▄▄▆█▅
accuracy_all,0.6154
accuracy_chinese,0.6207
accuracy_english,0.6098
avg_tokens_per_sample,18006.20316
input_tokens,5479
macro_f1_all,0.6122


#fxies

In [ ]:
# ===== Clean the results CSV and upload to wandb =====

import pandas as pd
import string
import wandb
from collections import Counter

# Configuration - UPDATE THESE
results_csv_path = "/content/results_incontext_audio_gemini-3-pro-preview_20260207-201631 (1).csv"
wandb_run_id = "j6kvv7o9"
wandb_project = "zero-shot-mci-classification"
cleaned_csv_filename = "results_incontext_audio_gemini-3-pro-preview_20260207-201631.csv"

# Load results
df = pd.read_csv(results_csv_path)
print(f"✓ Loaded {len(df)} samples\n")

# Function to clean predictions
def clean_prediction(text):
    """Remove LABEL: prefixes and extract just NC or MCI"""
    if pd.isna(text):
        return None

    text = str(text).upper().strip()

    # Check for explicit "LABEL:" pattern first
    if "LABEL: NC" in text:
        return "NC"
    if "LABEL: MCI" in text:
        return "MCI"

    # Remove punctuation
    text_clean = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text_clean.split()

    if not tokens:
        return None

    # Check the last word, which is typically the label
    last_word = tokens[-1]
    if last_word in ["NC", "MCI"]:
        return last_word

    return None

# Clean all individual prediction columns
print("Cleaning individual prediction columns...")
pred_cols = ['prediction_1', 'prediction_2', 'prediction_3']

for col in pred_cols:
    if col in df.columns:
        # Store original values for comparison
        original_values = df[col].copy()

        # Clean the predictions
        df[col] = df[col].apply(clean_prediction)

        # Count how many were changed
        changed = (original_values != df[col]).sum()
        print(f"  {col}: {changed} predictions cleaned")

# Recompute majority vote
print("\nRecomputing majority vote...")
def compute_majority_vote(row):
    """Compute majority vote from first 3 predictions"""
    votes = []
    for col in pred_cols:
        if col in df.columns and pd.notna(row[col]):
            votes.append(row[col])

    if not votes:
        return None

    # Use Counter for majority voting
    counter = Counter(votes)
    majority = counter.most_common(1)[0][0]

    return majority

# Store old prediction for comparison
old_prediction = df['prediction'].copy()

# Compute new majority vote
df['prediction'] = df.apply(compute_majority_vote, axis=1)

# Count how many majority votes changed
changed_mv = (old_prediction != df['prediction']).sum()
print(f"  Majority vote: {changed_mv} predictions changed")

# Show examples of what changed
if changed_mv > 0:
    print("\nExamples of changed majority votes:")
    changes = df[old_prediction != df['prediction']].head(5)
    for idx, row in changes.iterrows():
        print(f"  {row['tkdname']}: {old_prediction[idx]} → {row['prediction']}")

# Save cleaned CSV
cleaned_csv_path = f"/content/{cleaned_csv_filename}"
df.to_csv(cleaned_csv_path, index=False)
print(f"\n✓ Cleaned CSV saved to: {cleaned_csv_path}")

# Upload to wandb
print("\nUploading to wandb...")
run = wandb.init(
    project=wandb_project,
    id=wandb_run_id,
    resume="must"
)

# Create artifact
cleaned_artifact = wandb.Artifact(
    name=f"results_cleaned_{wandb_run_id}",
    type="cleaned_results",
    description="Results CSV with cleaned predictions (LABEL: prefixes removed) and recomputed majority votes"
)

# Add the cleaned CSV
cleaned_artifact.add_file(cleaned_csv_path, name=cleaned_csv_filename)

# Log the artifact
wandb.log_artifact(cleaned_artifact)

print(f"✓ Cleaned results uploaded to wandb as artifact")

wandb.finish()

print("\n" + "="*60)
print("CLEANING SUMMARY")
print("="*60)
print(f"Total samples: {len(df)}")
print(f"Individual predictions cleaned: {sum([changed for col in pred_cols])}")
print(f"Majority votes changed: {changed_mv}")
print(f"Final valid predictions: {df['prediction'].notna().sum()}")
print("\n✓ Done! Your cleaned CSV is now in wandb artifacts.")

✓ Loaded 507 samples

Cleaning individual prediction columns...
  prediction_1: 22 predictions cleaned
  prediction_2: 26 predictions cleaned
  prediction_3: 22 predictions cleaned

Recomputing majority vote...
  Majority vote: 24 predictions changed

Examples of changed majority votes:
  taukdial-004-2.wav: NC → MCI
  taukdial-032-3.wav: NC → MCI
  taukdial-035-1.wav: NC → MCI
  taukdial-052-1.wav: MCI → NC
  taukdial-065-3.wav: MCI → NC

✓ Cleaned CSV saved to: /content/results_incontext_audio_gemini-3-pro-preview_20260207-201631.csv

Uploading to wandb...


✓ Cleaned results uploaded to wandb as artifact


accuracy_all,0.6134
accuracy_chinese,0.6743
accuracy_english,0.5488
avg_tokens_per_sample,17990.10059
input_tokens,6480
macro_f1_all,0.6106
macro_f1_chinese,0.6734
macro_f1_english,0.5424
micro_f1_all,0.6134
micro_f1_chinese,0.6743
+14,...



CLEANING SUMMARY
Total samples: 507
Individual predictions cleaned: 66
Majority votes changed: 24
Final valid predictions: 507

✓ Done! Your cleaned CSV is now in wandb artifacts.


In [ ]:
# ===== Corrected evaluation with proper majority voting =====

import pandas as pd
import numpy as np
import string
import wandb
from sklearn.metrics import accuracy_score, recall_score, f1_score

# Configuration - UPDATE THESE
results_csv_path = "/content/results_incontext_audio_gemini-3-pro-preview_20260207-201631 (1).csv"
wandb_run_id = "j6kvv7o9"
wandb_project = "zero-shot-mci-classification"

# Load results
df = pd.read_csv(results_csv_path)
print(f"✓ Loaded {len(df)} samples\n")

# Function to extract 'NC' or 'MCI' from the prediction text
def clean_prediction(text):
    if pd.isna(text):
        return None

    text = str(text).upper().strip()

    # Check for explicit "LABEL:" pattern first
    if "LABEL: NC" in text:
        return "NC"
    if "LABEL: MCI" in text:
        return "MCI"

    # Remove punctuation
    text_clean = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text_clean.split()

    if not tokens:
        return None

    # Check the last word, which is typically the label
    last_word = tokens[-1]
    if last_word in ["NC", "MCI"]:
        return last_word

    return None

# Clean the first 3 prediction columns
pred_cols = ['prediction_1', 'prediction_2', 'prediction_3']
for col in pred_cols:
    df[f'{col}_clean'] = df[col].apply(clean_prediction)

# Majority Voting Function
def get_majority_vote(row):
    votes = [row[f'{col}_clean'] for col in pred_cols if row[f'{col}_clean'] is not None]

    if not votes:
        return None

    nc_count = votes.count('NC')
    mci_count = votes.count('MCI')

    if nc_count > mci_count:
        return 'NC'
    elif mci_count > nc_count:
        return 'MCI'
    else:
        return votes[0] if votes else None

# Apply majority voting
df['mv_prediction'] = df.apply(get_majority_vote, axis=1)

# Compute metrics by language subset
def calculate_metrics(subset_df, subset_name):
    subset_valid = subset_df.dropna(subset=['mv_prediction'])

    if len(subset_valid) == 0:
        return None

    y_true = subset_valid['dx']
    y_pred = subset_valid['mv_prediction']

    return {
        'subset': subset_name,
        'samples': len(subset_df),
        'valid_samples': len(subset_valid),
        'accuracy': round(accuracy_score(y_true, y_pred), 4),
        'uar': round(recall_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'micro_f1': round(f1_score(y_true, y_pred, average='micro', zero_division=0), 4),
        'macro_f1': round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4)
    }

# Calculate for all subsets
all_data = calculate_metrics(df, "all_data")
english_data = df[df['language'].astype(str).str.lower().isin(['en', 'english'])]
english_metrics = calculate_metrics(english_data, "english_only")
chinese_data = df[df['language'].astype(str).str.lower().isin(['zh', 'chinese', 'cn'])]
chinese_metrics = calculate_metrics(chinese_data, "chinese_only")

# Print results
print(f"📊 CORRECTED EVALUATION RESULTS:")
print(f"{'Subset':<15} {'Valid/Total':<12} {'Accuracy':<10} {'UAR':<10} {'Micro F1':<10} {'Macro F1':<10}")
print("-" * 75)
for metrics in [all_data, english_metrics, chinese_metrics]:
    if metrics:
        sample_str = f"{metrics['valid_samples']}/{metrics['samples']}"
        print(f"{metrics['subset']:<15} {sample_str:<12} {metrics['accuracy']:<10} {metrics['uar']:<10} {metrics['micro_f1']:<10} {metrics['macro_f1']:<10}")

# Resume wandb run and push corrected metrics
run = wandb.init(
    project=wandb_project,
    id=wandb_run_id,
    resume="must"
)

# Update wandb with corrected metrics
wandb.run.summary.update({
    # All data
    "accuracy_all": all_data["accuracy"],
    "uar_all": all_data["uar"],
    "micro_f1_all": all_data["micro_f1"],
    "macro_f1_all": all_data["macro_f1"],
    "valid_samples_all": all_data["valid_samples"],

    # English only
    "accuracy_english": english_metrics["accuracy"],
    "uar_english": english_metrics["uar"],
    "micro_f1_english": english_metrics["micro_f1"],
    "macro_f1_english": english_metrics["macro_f1"],
    "valid_samples_english": english_metrics["valid_samples"],

    # Chinese only
    "accuracy_chinese": chinese_metrics["accuracy"],
    "uar_chinese": chinese_metrics["uar"],
    "micro_f1_chinese": chinese_metrics["micro_f1"],
    "macro_f1_chinese": chinese_metrics["macro_f1"],
    "valid_samples_chinese": chinese_metrics["valid_samples"]
})

print("\n✓ Corrected metrics pushed to wandb!")
wandb.finish()

print(f"\n🎉 Final UAR: {all_data['uar']} (should be ~0.63)")

✓ Loaded 507 samples

📊 CORRECTED EVALUATION RESULTS:
Subset          Valid/Total  Accuracy   UAR        Micro F1   Macro F1  
---------------------------------------------------------------------------
all_data        507/507      0.6134     0.6337     0.6134     0.6106    
english_only    246/246      0.5488     0.6141     0.5488     0.5424    
chinese_only    261/261      0.6743     0.675      0.6743     0.6734    



✓ Corrected metrics pushed to wandb!


accuracy_all,0.6134
accuracy_chinese,0.6743
accuracy_english,0.5488
avg_tokens_per_sample,17990.10059
input_tokens,6480
macro_f1_all,0.6106
macro_f1_chinese,0.6734
macro_f1_english,0.5424
micro_f1_all,0.6134
micro_f1_chinese,0.6743
+14,...



🎉 Final UAR: 0.6337 (should be ~0.63)


In [ ]:
# ===== Re-evaluate and push corrected metrics to wandb =====

import pandas as pd
import wandb
from sklearn.metrics import accuracy_score, f1_score, recall_score

# Configuration - UPDATE THESE
results_csv_path = "/content/results_incontext_audio_gemini-3-pro-preview_20260207-201631 (1).csv"  # Your uploaded CSV path
wandb_run_id = "j6kvv7o9"  # Your existing run ID
wandb_project = "zero-shot-mci-classification"  # Your project name

# Load results
results_df = pd.read_csv(results_csv_path)
print(f"✓ Loaded {len(results_df)} samples")

# Resume the wandb run
run = wandb.init(
    project=wandb_project,
    id=wandb_run_id,
    resume="must",  # Must resume existing run
)
print(f"✓ Resumed wandb run: {wandb_run_id}")

# Call your fixed evaluate_results function
metrics = evaluate_results(results_df, run_name="recomputed")

# Optional: Print summary
print("\n" + "="*60)
print("CORRECTED METRICS PUSHED TO WANDB")
print("="*60)
print(f"All Data UAR: {metrics['all_data']['uar']}")
print(f"English UAR: {metrics['english_only']['uar']}")
print(f"Chinese UAR: {metrics['chinese_only']['uar']}")

# Finish the run
wandb.finish()
print("\n✓ Done! Check your wandb dashboard for updated metrics.")

✓ Loaded 507 samples


✓ Resumed wandb run: j6kvv7o9

📊 EVALUATION RESULTS:
Subset          Valid/Total  Accuracy   UAR        Micro F1   Macro F1  
---------------------------------------------------------------------------
all_data        507/507      0.6055     0.6257     0.6055     0.6027    
english_only    246/246      0.5325     0.5968     0.5325     0.5259    
chinese_only    261/261      0.6743     0.675      0.6743     0.6734    
✓ Metrics logged to W&B

CORRECTED METRICS PUSHED TO WANDB
All Data UAR: 0.6257
English UAR: 0.5968
Chinese UAR: 0.675


accuracy_all,0.6055
accuracy_chinese,0.6743
accuracy_english,0.5325
avg_tokens_per_sample,17990.10059
input_tokens,6480
macro_f1_all,0.6027
macro_f1_chinese,0.6734
macro_f1_english,0.5259
micro_f1_all,0.6055
micro_f1_chinese,0.6743
+14,...



✓ Done! Check your wandb dashboard for updated metrics.


## testing

In [ ]:
bucket_name = "taukadial-25"
groundtruth_path = "groundtruth/groundtruth_combined_lang2.csv"
transcription_path = "text-data/transcription_data_modified.csv"
model_name = "gemini-3-pro-preview"
collect_reasoning = False
temperature = 1
classification_prompt = """
Assess the cognitive condition based on the input audio and text data, where an elderly speaker describes one of three images as part of a clinician-guided task.
Indicate the diagnosis using one of these labels: NC (Normal Cognitive) or MCI (Mild Cognitive Impairment).
Output only NC or MCI as your response. Do not include any explanation, reasoning, or additional text.
"""
classification_type = "multimodal" # audio, text, or multimodal, make sure to change the prompt to reflect this
timestamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
run_name = f"{classification_type}_{model_name}_{timestamp}"

# GCS initialization
storage_client = storage.Client()
gcs_bucket = storage_client.bucket(bucket_name)

# Gemini Client initialization
client = genai.Client(
    #api_key = userdata.get('GOOGLE_API_KEY'),
    vertexai = True,
    project = "grand-store-465802-r4",
    location = "global"
)

from io import StringIO
blob = gcs_bucket.blob(groundtruth_path)
content = blob.download_as_text()
results_df = pd.read_csv(StringIO(content))

# load transcription data & create lookup map, tehcnically only needed for text & multimodal
transcription_blob = gcs_bucket.blob(transcription_path)
transcription_content = transcription_blob.download_as_text()
transcription_df = pd.read_csv(StringIO(transcription_content))
transcription_map = transcription_df.set_index('tkdname')['transcription'].to_dict()

results_df['prediction'] = None
results_df['response_time'] = None
results_df['finish_reason'] = None
results_df['input_tokens'] = None
results_df['output_tokens'] = None
results_df['thoughts_tokens'] = None
results_df['total_tokens'] = None

# Initialize reasoning dictionary
reasoning_dict = {} if collect_reasoning else None

# Randomly select a target sample
target_tkdname = results_df.sample(n=1)['tkdname'].iloc[0] # Changed to randomly select

# Get target sample info
target_row = results_df[results_df['tkdname'] == target_tkdname].iloc[0]
tkdname = target_row['tkdname']
language = target_row['language']
transcription = transcription_map.get(tkdname)

print(f"Processing sample: {tkdname}")
print(f"Language: {language}")
print(f"True label: {target_row['dx']}")
print(f"Collect reasoning: {collect_reasoning}")

# Filter available pool of exemplars, exemplar != target, and language matches
available_pool = results_df[
    (results_df['tkdname'] != tkdname) &
    (results_df['language'].astype(str).str.lower() == language.lower())
]

# Separate by class for sampling
nc_samples = available_pool[available_pool['dx'] == 'NC']
mci_samples = available_pool[available_pool['dx'] == 'MCI']

print(f"Available NC exemplars: {len(nc_samples)}")
print(f"Available MCI exemplars: {len(mci_samples)}")

predictions = []
total_tokens_used = 0
total_runtime = 0

for i in range(5):
    #print(f"\n{'='*60}")
    print(f"ITERATION {i+1}")
    #print('='*60)

    # Sample exemplars
    nc_exemplar = nc_samples.sample(n=1).iloc[0]
    mci_exemplar = mci_samples.sample(n=1).iloc[0]

    print(f"NC exemplar: {nc_exemplar['tkdname']}")
    print(f"MCI exemplar: {mci_exemplar['tkdname']}")

    exemplars = [
        {
            'tkdname': nc_exemplar['tkdname'],
            'label': nc_exemplar['dx'],
            'transcription': transcription_map.get(nc_exemplar['tkdname']),
            'language': nc_exemplar['language']
        },
        {
            'tkdname': mci_exemplar['tkdname'],
            'label': mci_exemplar['dx'],
            'transcription': transcription_map.get(mci_exemplar['tkdname']),
            'language': mci_exemplar['language']
        }
    ]

    # Call classification
    result = classify_with_exemplars(
            client,
            model_name,
            classification_prompt,
            classification_type,
            collect_reasoning,
            temperature,
            tkdname,
            transcription,
            exemplars
        )

    print(f"\nRESULTS:")
    print(f"  Prediction: {result['prediction']}")
    print(f"  Response time: {result['response_time']}s")
    print(f"  Total tokens: {result['total_tokens']}")
    print(f"  Thoughts tokens: {result['thoughts_tokens']}")

    # Check if reasoning was collected
    if collect_reasoning:
        reasoning_text = result.get('reasoning_text')
        if reasoning_text:
            print(f"\n  ✓ REASONING COLLECTED (Length: {len(reasoning_text)} chars)")
            print(f"  First 200 chars of reasoning:")
            print(f"  {reasoning_text[:200]}{'...' if len(reasoning_text) > 200 else ''}")

            # Store in reasoning dict
            if tkdname not in reasoning_dict:
                reasoning_dict[tkdname] = {}
            reasoning_dict[tkdname][f'iteration_{i+1}'] = reasoning_text
        else:
            print(f"\n  ⚠️  NO REASONING COLLECTED")

    predictions.append(result['prediction'])
    total_tokens_used += result['total_tokens']
    total_runtime += result['response_time']

    #print(f"\n  Running predictions so far: {predictions}")


# Majority vote
def majority_vote(predictions):
    from collections import Counter
    if not predictions:
        return "ERROR"
    counts = Counter(predictions)
    # Strip 'LABEL: ' prefix and any leading/trailing whitespace
    most_common = counts.most_common(1)[0][0]
    if most_common.startswith('LABEL:'):
        return most_common.replace('LABEL:', '').strip()
    return most_common.strip()

final_prediction = majority_vote(predictions)

# Clean true label for comparison
cleaned_true_label = target_row['dx'].strip()

print(f"\n" + "="*60)
print(f"FINAL RESULTS")
print("="*60)
print(f"Individual predictions: {predictions}")
print(f"Final prediction (majority vote): {final_prediction}")
print(f"True label: {cleaned_true_label}") # Use cleaned true label
print(f"Correct: {final_prediction == cleaned_true_label}") # Compare cleaned values
print(f"Total tokens used across 5 iterations: {total_tokens_used}")
print(f"Total runtime across 5 iterations: {total_runtime:.2f}s")

if collect_reasoning and reasoning_dict:
    print(f"\nREASONING SUMMARY:")
    print(f"Reasoning collected for sample: {tkdname}")
    print(f"Number of iterations with reasoning: {len(reasoning_dict[tkdname])}")
    for iteration, reasoning in reasoning_dict[tkdname].items():
        print(f"  {iteration}: {len(reasoning)} characters of reasoning")
else:
    print(f"\nNo reasoning collected (collect_reasoning={collect_reasoning})")

print("="*60)


Processing sample: taukdial-108-1.wav
Language: zh
True label: MCI
Collect reasoning: False
Available NC exemplars: 129
Available MCI exemplars: 131
ITERATION 1
NC exemplar: taukdial-066-1.wav
MCI exemplar: taukdial-051-1.wav


KeyboardInterrupt: 

In [ ]:
# Quick verification script for save_results and evaluate_results functions
import pandas as pd
import json
import os
from datetime import datetime

# Create a small test dataset that mimics your real data structure
test_data = {
    'tkdname': ['test001', 'test002', 'test003', 'test004', 'test005'],
    'dx': ['NC', 'MCI', 'NC', 'MCI', 'NC'],  # True labels
    'language': ['en', 'en', 'zh', 'zh', 'en'],
    'prediction': ['NC', 'MCI', 'NC', 'ERROR', 'MCI'],  # Mix of correct, incorrect, and error
    'prediction_1': ['NC', 'MCI', 'NC', 'ERROR', 'MCI'],
    'response_time_1': [1.2, 1.5, 1.8, 0.0, 1.3],
    'finish_reason_1': ['STOP', 'STOP', 'STOP', 'ERROR', 'STOP'],
    'input_tokens_1': [100, 105, 110, 0, 95],
    'output_tokens_1': [5, 5, 5, 0, 5],
    'thoughts_tokens_1': [50, 45, 55, 0, 48],
    'total_tokens_1': [155, 155, 170, 0, 148]
}

test_df = pd.DataFrame(test_data)

# Create test reasoning dict
test_reasoning_dict = {
    'test001': {'iteration_1': 'This sample shows clear cognitive patterns...'},
    'test002': {'iteration_1': 'The audio indicates some mild impairment...'},
    'test003': {'iteration_1': '这个样本显示正常的认知模式...'}
}

print("🔍 TESTING SAVE_RESULTS AND EVALUATE_RESULTS")
print("="*60)

# Test 1: Check if save_results works
print("\n1️⃣ Testing save_results function...")
try:
    test_run_name = f"test_run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    # Check if we're in the expected directory
    current_dir = os.getcwd()
    print(f"Current directory: {current_dir}")

    # Test without WandB (comment out wandb parts temporarily)
    print("Testing CSV save...")

    # Modify the save path to current directory instead of /content/
    csv_filename = f"results_{test_run_name}.csv"
    csv_path = os.path.join(current_dir, csv_filename)
    test_df.to_csv(csv_path, index=False)
    print(f"✅ CSV saved successfully to: {csv_path}")

    # Test reasoning JSON save
    if test_reasoning_dict:
        json_filename = f"reasoning_{test_run_name}.json"
        json_path = os.path.join(current_dir, json_filename)
        with open(json_path, 'w') as f:
            json.dump(test_reasoning_dict, f, indent=2)
        print(f"✅ Reasoning JSON saved successfully to: {json_path}")

    # Clean up test files
    os.remove(csv_path)
    os.remove(json_path)
    print("✅ Test files cleaned up")

except Exception as e:
    print(f"❌ save_results test failed: {str(e)}")

# Test 2: Check if evaluate_results works
print("\n2️⃣ Testing evaluate_results function...")
try:
    # Test the evaluation logic without wandb
    from sklearn.metrics import accuracy_score, f1_score, recall_score

    def test_calculate_subset_metrics(subset_df, subset_name):
        # Filter out errors
        valid_df = subset_df[subset_df['prediction'] != "ERROR"]
        print(f"  {subset_name}: {len(valid_df)}/{len(subset_df)} valid samples")

        if len(valid_df) == 0:
            print(f"  ⚠️ No valid samples for {subset_name}")
            return None

        y_true = valid_df['dx'].values
        y_pred = valid_df['prediction'].values

        print(f"  True labels: {list(y_true)}")
        print(f"  Predictions: {list(y_pred)}")

        return {
            "subset": subset_name,
            "samples": len(subset_df),
            "valid_samples": len(valid_df),
            "accuracy": round(accuracy_score(y_true, y_pred), 4),
            "uar": round(recall_score(y_true, y_pred, average='macro', zero_division=0), 4),
            "micro_f1": round(f1_score(y_true, y_pred, average='micro', zero_division=0), 4),
            "macro_f1": round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4)
        }

    # Test all data
    all_metrics = test_calculate_subset_metrics(test_df, "all_data")

    # Test English subset
    english_data = test_df[test_df['language'].astype(str).str.lower().isin(['en', 'english'])]
    english_metrics = test_calculate_subset_metrics(english_data, "english_only")

    # Test Chinese subset
    chinese_data = test_df[test_df['language'].astype(str).str.lower().isin(['zh', 'chinese', 'cn'])]
    chinese_metrics = test_calculate_subset_metrics(chinese_data, "chinese_only")

    print("\n📊 EVALUATION RESULTS:")
    print(f"{'Subset':<15} {'Valid/Total':<12} {'Accuracy':<10} {'UAR':<10} {'Micro F1':<10} {'Macro F1':<10}")
    print("-" * 75)

    for metrics in [all_metrics, english_metrics, chinese_metrics]:
        if metrics:
            sample_str = f"{metrics['valid_samples']}/{metrics['samples']}"
            print(f"{metrics['subset']:<15} {sample_str:<12} {metrics['accuracy']:<10} {metrics['uar']:<10} {metrics['micro_f1']:<10} {metrics['macro_f1']:<10}")

    print("✅ evaluate_results logic works correctly")

except Exception as e:
    print(f"❌ evaluate_results test failed: {str(e)}")

# Test 3: Check data structure compatibility
print("\n3️⃣ Testing data structure compatibility...")
print("Required columns for evaluate_results:")
required_cols = ['dx', 'prediction', 'language']
for col in required_cols:
    if col in test_df.columns:
        print(f"✅ Column '{col}' exists")
        print(f"   Unique values: {test_df[col].unique()}")
    else:
        print(f"❌ Column '{col}' missing!")

print(f"\n✅ Verification complete!")
print("\n🔧 RECOMMENDED FIXES:")
print("1. Update save_results to use current directory instead of /content/")
print("2. Make sure your real data has 'dx' column for true labels")
print("3. Consider adding error handling for empty subsets")
print("4. Test with your actual WandB setup before full run")

🔍 TESTING SAVE_RESULTS AND EVALUATE_RESULTS

1️⃣ Testing save_results function...
Current directory: /content
Testing CSV save...
✅ CSV saved successfully to: /content/results_test_run_20250818_002424.csv
✅ Reasoning JSON saved successfully to: /content/reasoning_test_run_20250818_002424.json
✅ Test files cleaned up

2️⃣ Testing evaluate_results function...
  all_data: 4/5 valid samples
  True labels: ['NC', 'MCI', 'NC', 'NC']
  Predictions: ['NC', 'MCI', 'NC', 'MCI']
  english_only: 3/3 valid samples
  True labels: ['NC', 'MCI', 'NC']
  Predictions: ['NC', 'MCI', 'MCI']
  chinese_only: 1/2 valid samples
  True labels: ['NC']
  Predictions: ['NC']

📊 EVALUATION RESULTS:
Subset          Valid/Total  Accuracy   UAR        Micro F1   Macro F1  
---------------------------------------------------------------------------
all_data        4/5          0.75       0.8333     0.75       0.7333    
english_only    3/3          0.6667     0.75       0.6667     0.6667    
chinese_only    1/2       

In [ ]:
# args
bucket_name = "taukadial-25"
groundtruth_path = "groundtruth/groundtruth_combined_lang2.csv"
transcription_path = "text-data/transcription_data_modified.csv"
model_name = "gemini-2.5-pro"
collect_reasoning = True
temperature = 1
classification_prompt = """
Assess the cognitive condition based on the input audio and text data, where an elderly speaker describes one of three images as part of a clinician-guided task.
Indicate the diagnosis using one of these labels: NC (Normal Cognitive) or MCI (Mild Cognitive Impairment).
Output only NC or MCI as your response. Do not include any explanation, reasoning, or additional text.
"""
classification_type = "multimodal" # audio, text, or multimodal, make sure to change the prompt to reflect this
timestamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
run_name = f"{classification_type}_{model_name}_{timestamp}"

# GCS initialization
storage_client = storage.Client()
gcs_bucket = storage_client.bucket(bucket_name)

# Gemini Client initialization
client = genai.Client(
    #api_key = userdata.get('GOOGLE_API_KEY'),
    vertexai = True,
    project = "grand-store-465802-r4",
    location = "us-west1"
)

from io import StringIO
blob = gcs_bucket.blob(groundtruth_path)
content = blob.download_as_text()
results_df = pd.read_csv(StringIO(content))

# load transcription data & create lookup map, tehcnically only needed for text & multimodal
transcription_blob = gcs_bucket.blob(transcription_path)
transcription_content = transcription_blob.download_as_text()
transcription_df = pd.read_csv(StringIO(transcription_content))
transcription_map = transcription_df.set_index('tkdname')['transcription'].to_dict()

results_df['prediction'] = None
results_df['response_time'] = None
results_df['finish_reason'] = None
results_df['input_tokens'] = None
results_df['output_tokens'] = None
results_df['thoughts_tokens'] = None
results_df['total_tokens'] = None

# Single sample ICL classification
target_tkdname = "taukdial-002-1.wav"  # Replace with your target sample

# Get target sample info
target_row = results_df[results_df['tkdname'] == target_tkdname].iloc[0]
tkdname = target_row['tkdname']
language = target_row['language']
transcription = transcription_map.get(tkdname)

print(f"Processing sample: {tkdname}")
print(f"Language: {language}")
print(f"True label: {target_row['dx']}")

# Filter available pool of exemplars, exemplar != target, and language matches
available_pool = results_df[
    (results_df['tkdname'] != tkdname) &
    (results_df['language'].astype(str).str.lower() == language.lower())
]

# Separate by class for sampling
nc_samples = available_pool[available_pool['dx'] == 'NC']
mci_samples = available_pool[available_pool['dx'] == 'MCI']

print(f"Available NC exemplars: {len(nc_samples)}")
print(f"Available MCI exemplars: {len(mci_samples)}")

predictions = []
total_tokens_used = 0
total_runtime = 0

for i in range(5):
    print(f"\nIteration {i+1}:")

    # Sample exemplars
    nc_exemplar = nc_samples.sample(n=1).iloc[0]
    mci_exemplar = mci_samples.sample(n=1).iloc[0]

    print(f"  NC exemplar: {nc_exemplar['tkdname']}")
    print(f"  MCI exemplar: {mci_exemplar['tkdname']}")

    exemplars = [
        {
            'tkdname': nc_exemplar['tkdname'],
            'label': nc_exemplar['dx'],
            'transcription': transcription_map.get(nc_exemplar['tkdname']),
            'language': nc_exemplar['language']
        },
        {
            'tkdname': mci_exemplar['tkdname'],
            'label': mci_exemplar['dx'],
            'transcription': transcription_map.get(mci_exemplar['tkdname']),
            'language': mci_exemplar['language']
        }
    ]

    # Call classify_with_exemplars
    result = classify_with_exemplars(
            client,
            model_name,
            classification_prompt,
            classification_type,
            collect_reasoning,
            temperature,
            tkdname,
            transcription,
            exemplars
        )

    print(f"  Prediction: {result['prediction']}")
    print(f"  Response time: {result['response_time']}s")
    print(f"  Total tokens: {result['total_tokens']}")

    predictions.append(result['prediction'])
    total_tokens_used += result['total_tokens']
    total_runtime += result['response_time']


# Majority vote
def majority_vote(predictions):
    from collections import Counter
    if not predictions:
        return "ERROR"
    counts = Counter(predictions)
    return counts.most_common(1)[0][0]

final_prediction = majority_vote(predictions)

print(f"\n" + "="*50)
print(f"RESULTS:")
print(f"Individual predictions: {predictions}")
print(f"Final prediction (majority vote): {final_prediction}")
print(f"True label: {target_row['dx']}")
print(f"Correct: {final_prediction == target_row['dx']}")
print(f"Total tokens used across 5 iterations: {total_tokens_used}")
print(f"Total runtime across 5 iterations: {total_runtime:.2f}s")
print("="*50)

Processing sample: taukdial-002-1.wav
Language: en
True label: NC
Available NC exemplars: 92
Available MCI exemplars: 153

Iteration 1:
  NC exemplar: taukdial-075-3.wav
  MCI exemplar: taukdial-163-1.wav
  Prediction: MCI
  Response time: 20.02s
  Total tokens: 7808

Iteration 2:
  NC exemplar: taukdial-113-1.wav
  MCI exemplar: taukdial-046-2.wav
  Prediction: NC
  Response time: 17.83s
  Total tokens: 7694

Iteration 3:
  NC exemplar: taukdial-078-2.wav
  MCI exemplar: taukdial-138-3.wav
  Prediction: NC
  Response time: 14.24s
  Total tokens: 8415

Iteration 4:
  NC exemplar: taukdial-008-2.wav
  MCI exemplar: taukdial-029-2.wav
  Prediction: NC
  Response time: 20.73s
  Total tokens: 10201

Iteration 5:
  NC exemplar: taukdial-085-3.wav
  MCI exemplar: taukdial-163-3.wav
  Prediction: MCI
  Response time: 18.92s
  Total tokens: 6755

RESULTS:
Individual predictions: ['MCI', 'NC', 'NC', 'NC', 'MCI']
Final prediction (majority vote): NC
True label: NC
Correct: True
Total tokens used

In [ ]:
# Given a tkdname, find matching language exemplars and construct exemplar list

target_tkdname = "taukdial-002-1.wav"

# Get target sample language
target_row = results_df[results_df['tkdname'] == target_tkdname].iloc[0]
target_language = target_row['language']

# Filter available samples (exclude target, match language)
available_pool = results_df[
    (results_df['tkdname'] != target_tkdname) &
    (results_df['language'].astype(str).str.lower() == target_language.lower())
]

# Separate by class and sample one from each
nc_samples = available_pool[available_pool['dx'] == 'NC']
mci_samples = available_pool[available_pool['dx'] == 'MCI']

nc_exemplar = nc_samples.sample(n=1).iloc[0]
mci_exemplar = mci_samples.sample(n=1).iloc[0]

# Construct exemplar list dictionary
exemplars = [
    {
        'tkdname': nc_exemplar['tkdname'],
        'label': nc_exemplar['dx'],
        'transcription': transcription_map.get(nc_exemplar['tkdname']),
        'language': nc_exemplar['language']
    },
    {
        'tkdname': mci_exemplar['tkdname'],
        'label': mci_exemplar['dx'],
        'transcription': transcription_map.get(mci_exemplar['tkdname']),
        'language': mci_exemplar['language']
    }
]